# Tacotron — Towards End-to-End Speech Synthesis (2017)

_Papers · Speech & Audio_

**Map raw characters straight to a spectrogram with one attention seq2seq, then turn the spectrogram into sound with Griffin-Lim — no hand-built text-to-speech pipeline.**

---

This notebook is a practice scaffold for the **Tacotron — Towards End-to-End Speech Synthesis (2017)** lesson. We build it up one piece at a time: a worked attention step, the attention block, the char encoder and frame decoder, then a tiny char→frame model with a clean fixed-context ablation. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — Hand-check one attention step

Tacotron's decoder uses content-based (tanh) attention, the same form as Bahdanau. For each decoder state `s`, a score `e_j = v^T tanh(W s + U h_j)` is computed against every character annotation `h_j`, softmax turns the scores into weights `alpha`, and the context `c` is the weighted sum of annotations. We work one step by hand on three 2-dim annotations so the score → softmax → context pipeline is concrete before we wrap it in a module.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

# --- Worked example: score -> softmax -> context, by hand. ---
h = torch.tensor([[1., 0.], [0., 1.], [1., 1.]])    # 3 char annotations h_j (2-dim)
s = torch.tensor([0.5, -0.5])                        # decoder state s_{i-1}
W = torch.tensor([[0.5, 0.], [0., 0.5]])
U = torch.eye(2)
v = torch.tensor([1., 1.])

e = (v * torch.tanh(W @ s + h @ U.T)).sum(1)         # e_j = v^T tanh(W s + U h_j)
alpha = torch.softmax(e, dim=0)
c = (alpha.unsqueeze(-1) * h).sum(0)                 # weighted sum of annotations

print("e     =", [round(x, 4) for x in e.tolist()])      # [0.6034, 0.8801, 1.4834]
print("alpha =", [round(x, 4) for x in alpha.tolist()])  # [0.2114, 0.2788, 0.5098]
print("context c =", [round(x, 4) for x in c.tolist()])  # [0.7212, 0.7886]

### Step 2 — Build the attention block

Now the same calculation as a batched module. `W` projects the decoder state, `U` projects each annotation, and `v` collapses the tanh to a scalar score per character. Softmax normalizes **over characters**, and the context is the weighted sum of annotations. This is the content-based attention of Tacotron §3.3 — identical in form to Bahdanau. We also fix the toy dimensions used throughout: a small vocab, a few frames per character, and the reduction factor `r`.

In [ ]:
# --- Content-based (tanh) attention block (Tacotron \S 3.3; same form as Bahdanau). ---
VOCAB, FPC, FRAME, R, EMB, HID = 6, 3, 4, 1, 16, 32   # vocab, frames-per-char, frame dim, reduction r
NCHAR = 5                                              # chars per toy utterance
NFRAME = NCHAR * FPC                                   # total frames per utterance (= 15)

class Attention(nn.Module):
    def __init__(self, hid, ann):
        super().__init__()
        self.W = nn.Linear(hid, hid, bias=False)      # W s_{i-1}
        self.U = nn.Linear(ann, hid, bias=False)      # U h_j
        self.v = nn.Linear(hid, 1, bias=False)        # v^T
    def forward(self, s, H):                           # s:(N,hid)  H:(N,T_x,ann)
        e = self.v(torch.tanh(self.W(s).unsqueeze(1) + self.U(H))).squeeze(-1)  # score -> (N,T_x)
        alpha = torch.softmax(e, dim=1)               # normalize over CHARACTERS
        c = (alpha.unsqueeze(-1) * H).sum(1)          # weighted-sum context -> (N,ann)
        return c, alpha

### Step 3 — Build the char encoder and frame decoder

The encoder embeds characters and runs a bidirectional GRU to produce one annotation per character (standing in for the paper's CBHG). The decoder is a GRU cell that, at each step, takes the previous frame plus a context vector and predicts `R` frames at once. The `attend` flag chooses between **fresh per-step attention** (the real model) and a **fixed-context ablation** that reuses one annotation for every step — the knob for the experiment in Step 4.

In [ ]:
# --- Char encoder + frame decoder. attend=False -> fixed-context ablation. ---
class Encoder(nn.Module):                              # stands in for the paper's CBHG
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(VOCAB, EMB)
        self.rnn = nn.GRU(EMB, HID, batch_first=True, bidirectional=True)
    def forward(self, x):
        return self.rnn(self.emb(x))[0]               # annotations -> (N, T_x, 2*HID)

class Decoder(nn.Module):
    def __init__(self, attend=True):
        super().__init__()
        self.attend = attend
        self.attn = Attention(HID, 2 * HID)
        self.cell = nn.GRUCell(FRAME + 2 * HID, HID)  # in: prev frame + context
        self.out = nn.Linear(HID, FRAME * R)          # predict R frames per step
    def forward(self, H):
        N = H.size(0)
        s = torch.zeros(N, HID)
        prev = torch.zeros(N, FRAME)                  # <GO>: all-zero start frame
        frames, attns = [], []
        for _ in range(NFRAME // R):                  # NFRAME/R decoder steps
            if self.attend:
                c, a = self.attn(s, H)                # fresh context EVERY step
            else:
                c = H[:, -1, :]                       # ABLATION: one fixed vector
                a = torch.zeros(N, NCHAR)
            s = self.cell(torch.cat([prev, c], -1), s)
            blk = self.out(s).view(N, R, FRAME)       # R frames at once
            frames.append(blk)
            attns.append(a)
            prev = blk[:, -1, :]                       # feed last frame of the block
        return torch.cat(frames, 1), torch.stack(attns, 1)   # (N,NFRAME,FRAME), (N,steps,T_x)

### Step 4 — Toy data, then train with and without attention

Each character owns a fixed random block of frames — its "pronunciation" — and a target utterance is just those blocks concatenated, so the model must learn to read the right character at the right time. We train twice, identical except for the `attend` flag. With attention, the L1 error is low and the average alignment is a near-diagonal monotonic band (decoder step `i` attends to char `i`); the fixed-context ablation has higher error and no alignment structure, because one vector cannot say which character the current frame belongs to.

In [ ]:
# --- Toy char->frame data: each char owns a fixed random frame-block; concat them. ---
torch.manual_seed(0)
CHAR_FRAMES = torch.randn(VOCAB, FPC, FRAME)           # the "pronunciation" of each character

def make(n):
    x = torch.randint(0, VOCAB, (n, NCHAR))            # char ids
    y = CHAR_FRAMES[x].reshape(n, NFRAME, FRAME)        # target frames = chars' blocks concatenated
    return x, y

def train(attend, epochs=40, N=2000):
    torch.manual_seed(0)
    enc, dec = Encoder(), Decoder(attend=attend)
    opt = torch.optim.Adam(list(enc.parameters()) + list(dec.parameters()), lr=3e-3)
    xb, yb = make(N)
    for ep in range(epochs):
        perm = torch.randperm(N)
        for i in range(0, N, 128):
            idx = perm[i:i + 128]
            pred, _ = dec(enc(xb[idx]))
            loss = (pred - yb[idx]).abs().mean()        # l1 loss on frames (paper \S 4)
            opt.zero_grad()
            loss.backward()
            opt.step()
    xt, yt = make(400)
    with torch.no_grad():
        pred, attns = dec(enc(xt))
        err = (pred - yt).abs().mean().item()
        A = attns.mean(0)                               # avg alignment (steps x chars)
    return err, A

err, A = train(attend=True)
print("\nATTENTION frame L1 error:", round(err, 4))
print("avg alignment (rows = decoder step, cols = char pos):")
for row in A.tolist():
    print("  ", [round(x, 3) for x in row])

err0, _ = train(attend=False)
print("\nFIXED-CONTEXT (ablation) frame L1 error:", round(err0, 4))
print("Attention gives a near-diagonal monotonic alignment and low error;")
print("the fixed-context model has higher error and no alignment structure.")
# (Exact numbers vary by hardware/seed; our small run, not the paper's reported result.)

## Visualize the data & results

_When a tiny char→spectrogram-frame attention seq2seq is trained on toy speech (each character owns a fixed frame-block), does the learned alignment α become the near-diagonal, MONOTONIC band that real Tacotron shows?_

### Step 1 — Rebuild a compact model that emits one frame per step

This panel is self-contained, so we re-declare the encoder, attention, and decoder tersely. The one difference from the reference implementation is `r = 1`: the decoder emits a single frame per step (`self.out` maps to `FRAME`), which makes the per-step alignment easiest to read as a heatmap. Re-seeding keeps the run reproducible.

In [ ]:
import torch
import torch.nn as nn

# Reproduces the qualitative effect: a learned, near-diagonal MONOTONIC char->frame alignment.
torch.manual_seed(0)
VOCAB, NCHAR, FPC, FRAME, EMB, HID, N = 6, 5, 3, 4, 16, 32, 2000
NFRAME = NCHAR * FPC

class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(VOCAB, EMB)
        self.rnn = nn.GRU(EMB, HID, batch_first=True, bidirectional=True)
    def forward(self, x):
        return self.rnn(self.emb(x))[0]

class Attention(nn.Module):
    def __init__(self):
        super().__init__()
        self.W = nn.Linear(HID, HID, bias=False)
        self.U = nn.Linear(2 * HID, HID, bias=False)
        self.v = nn.Linear(HID, 1, bias=False)
    def forward(self, s, H):
        e = self.v(torch.tanh(self.W(s).unsqueeze(1) + self.U(H))).squeeze(-1)  # content score
        a = torch.softmax(e, dim=1)                                             # over chars
        return (a.unsqueeze(-1) * H).sum(1), a                                  # context, weights

class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.attn = Attention()
        self.cell = nn.GRUCell(FRAME + 2 * HID, HID)
        self.out = nn.Linear(HID, FRAME)                # r = 1 frame per step here
    def forward(self, H):
        n = H.size(0)
        s = torch.zeros(n, HID)
        prev = torch.zeros(n, FRAME)
        fr, at = [], []
        for _ in range(NFRAME):
            c, a = self.attn(s, H)
            s = self.cell(torch.cat([prev, c], -1), s)
            prev = self.out(s)
            fr.append(prev)
            at.append(a)
        return torch.stack(fr, 1), torch.stack(at, 1)

### Step 2 — Train and read the learned alignment

We rebuild the toy char→frame data and train the model, then collapse the three frames of each character-block into one row so we can read the alignment at the character level. The result is near-diagonal and monotonic: frame step `i` attends to char `i`, exactly the band real Tacotron produces.

In [ ]:
CHAR_FRAMES = torch.randn(VOCAB, FPC, FRAME)            # each char's fixed frame-block

def make(n):
    x = torch.randint(0, VOCAB, (n, NCHAR))
    return x, CHAR_FRAMES[x].reshape(n, NFRAME, FRAME)

enc, dec = Encoder(), Decoder()
opt = torch.optim.Adam(list(enc.parameters()) + list(dec.parameters()), lr=3e-3)
xb, yb = make(N)
for ep in range(40):
    perm = torch.randperm(N)
    for i in range(0, N, 128):
        idx = perm[i:i + 128]
        pred, _ = dec(enc(xb[idx]))
        loss = (pred - yb[idx]).abs().mean()            # l1 frame loss (paper \S 4)
        opt.zero_grad()
        loss.backward()
        opt.step()

xt, yt = make(400)
with torch.no_grad():
    pred, attns = dec(enc(xt))
    err = (pred - yt).abs().mean().item()
    # collapse the 3 frames of each char-block into one row to read the char-level alignment:
    A = attns.mean(0).reshape(NCHAR, FPC, NCHAR).mean(1)
print("frame L1 error:", round(err, 4))
for row in A.tolist():
    print([round(x, 3) for x in row])
# Near-diagonal & monotonic: frame step i attends to char i. Our run, not the paper's numbers.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation. You have a working character &rarr; frame attention model whose alignment is a
            bright monotonic diagonal. Replace the per-step attention with a single fixed context (use the
            last encoder annotation for every decoder step, plain-seq2seq style) and retrain. What happens to the
            frame reconstruction error and to the alignment, and what does that demonstrate?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Swap the attention call for a fixed vector: set c_i = H[:, -1, :] for every output step, deleting the score/softmax/weighted-sum. — _An honest ablation changes exactly one thing &mdash; the attention &mdash; so any difference is attributable to it. This recreates the fixed-length bottleneck end-to-end TTS needs attention to avoid._
- Retrain with everything else identical and measure the frame $\ell_1$ error; also try to plot an alignment matrix. — _With no per-step weights there is no alignment to plot, and one vector must carry every character at once._
- Compare: the attention model reconstructs frames well with a near-diagonal heatmap; the fixed-context model has higher error (later frames worst) and no usable alignment. — _The single vector cannot say "which character does the current frame belong to," so the decoder cannot track its position in the text._

**Answer:** Frame error rises (later frames suffer most) and the alignment structure
                 disappears &mdash; there are no per-step weights to form a diagonal. Since the two models
                 differ only in "per-step attention vs one fixed context," this isolates attention as the reason
                 the decoder can track which character each frame belongs to. It reproduces Tacotron's premise:
                 attention is what lets a single seq2seq go from characters to a spectrogram.

</details>

**Problem 2.** Your worked example had $\alpha=[0.211,0.279,0.510]$ and annotations $h_1=[1,0]$, $h_2=[0,1]$,
            $h_3=[1,1]$, giving context $c=[0.721,0.789]$. Suppose at this step the scores were so extreme that
            softmax returned $\alpha=[0,0,1]$. What is the context, and what does that mean for TTS alignment?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Plug into the context sum: $c = 0\cdot[1,0]+0\cdot[0,1]+1\cdot[1,1] = [1,1]$. — _A one-hot $\alpha$ makes the context equal a single annotation &mdash; here $h_3$ exactly._
- Note this is "read exactly character 3" &mdash; a perfectly sharp, hard alignment for this frame. — _Softmax attention contains hard alignment as its sharp limit; a near-one-hot diagonal is exactly the picture a clean TTS model produces._
- Contrast with the real $\alpha=[0.211,0.279,0.510]$ giving $c=[0.721,0.789]$: a soft blend, mostly $h_3$. — _Early in training the weights are soft (gradients can flow); as it learns, each frame's row sharpens onto its character, sliding rightward step by step._

**Answer:** With $\alpha=[0,0,1]$ the context is $c=[1,1]=h_3$ &mdash; the frame reads exactly character
                 3. A trained Tacotron's alignment approaches a sharp diagonal of such near-one-hot rows that
                 moves forward as $i$ grows. So softmax attention is a soft, differentiable generalization of the
                 hard, monotonic character-to-frame alignment that speech needs.

</details>

**Problem 3.** The paper predicts $r$ frames per decoder step (it uses $r=2$). Suppose a toy clip has 12
            spectrogram frames. How many decoder steps run with $r=1$, $r=2$, and $r=4$, and what is the
            trade-off as $r$ grows?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Decoder steps $= \text{num frames} / r$. So $12/1=12$, $12/2=6$, $12/4=3$ steps. — _Each step emits a block of $r$ frames, so larger $r$ means proportionally fewer steps &mdash; the paper: predicting $r$ frames "divides the total number of decoder steps by $r$."_
- Note fewer steps means fewer attention computations and RNN unrollings: "reduces model size, training time, and inference time" (&sect;3.3). — _Attention and recurrence cost scale with the number of decoder steps, so dividing steps by $r$ is a direct speedup._
- Note the cost: each step must now predict $r$ frames at once from one state, and the alignment advances $r$ frames per step, so very large $r$ can blur fine timing. — _There is a sweet spot; the paper reports $r=2$ for its MOS result while noting larger $r$ (e.g. $r=5$) can also work._

**Answer:** With 12 frames: $r=1\Rightarrow 12$ steps, $r=2\Rightarrow 6$ steps, $r=4\Rightarrow 3$
                 steps. Larger $r$ divides the decoder steps (and so attention/RNN work) by $r$ &mdash; less
                 model size and training/inference time &mdash; at the cost of each step having to predict a
                 bigger frame block and the alignment advancing in coarser jumps. The paper uses $r=2$.

</details>